# Лабораторная работа №7: Vision и условная генерация

[Исходное задание](https://colab.research.google.com/drive/17eKtcwVyYMMIIyBa_kDFbVMn7BCNoTrP)

Выполнение через совместимый API RouterAI. API-ключ берётся из окружения или локального `.env`. Сохранённые результаты получены реальными запросами.



## Задание

##Задания на семинар
1. Есть названия 3-х объектов/предметов. Модель принимает изображение и формирует описание этого изображения. Далее LLM определяется присутствие всех 3-х объектов в описании (данный шаг реализовать с помощью цепочки chain). Если все объекты присутствуют, то генерируем изображение по полученному от LLM описанию. Если какие-то объекты отсутствуют, то генерируется изображение, содержащее только отсутствующие объекты.
4. Первая цепочка создает детальное текстовое описание сцены на изображении. Вторая ищет в описании указания на три стилевых признака (например, "готический", "футуристический", "ретро"). Если все три найдены, генерируется изображение по описанию в одном из этих стилей (случайно выбранном). Если нет, генерируется изображение в нейтральном стиле, но содержащее только те объекты, которые НЕ были связаны с искомыми стилями.

## Подключение

In [ ]:
from pathlib import Path
import sys

# При открытии из подпапки находим общий модуль конфигурации.
for candidate in [Path.cwd(), Path.cwd().parent]:
    if (candidate / "lab_runtime.py").exists():
        sys.path.insert(0, str(candidate))
        break

from lab_runtime import make_llm, json_result, ASSETS
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from IPython.display import display, Markdown, Image, Audio

llm = make_llm()
print("Модель:", llm.model_name)


## 1. Изображение → описание → проверка объектов

In [ ]:
from lab_runtime import generate_image, image_message
import random

source_image = generate_image("A photorealistic laptop, red mug and green potted plant on a wooden desk, daylight, no text.", "vision_input.png")
display(Image(filename=str(source_image), width=600))
describe_chain = ChatPromptTemplate.from_messages([
    ("system", "Опиши видимые предметы и признаки стиля на изображении. Не выдумывай невидимые объекты."),
    ("human", "{content}")
]) | llm | StrOutputParser()
# Изображение передаём как image_url внутри HumanMessage.
describe_chain = llm | StrOutputParser()
description = describe_chain.invoke([HumanMessage(content=image_message(source_image, "Детально опиши предметы, окружение и стили изображения по-русски."))])
print("Описание:", description)

objects_chain = ChatPromptTemplate.from_messages([
    ("system", 'Проверь объекты по описанию. Верни JSON {{"all_found":true/false,"missing":[]}}. Не придумывай объекты.'),
    ("human", "Описание: {description}\nОбъекты: {objects}")
]) | llm | StrOutputParser()
for index, objects in enumerate([["ноутбук", "кружка", "растение"], ["ноутбук", "гитара", "микрофон"]]):
    checked = json_result(objects_chain.invoke({"description": description, "objects": objects}))
    print("Требования:", objects, "Результат:", checked)
    prompt = description if checked["all_found"] else "Изображение только следующих предметов: " + ", ".join(checked["missing"]) + ". Нейтральный фон, без других объектов."
    output = generate_image(prompt, f"lab07_objects_{index}.png")
    display(Image(filename=str(output), width=450))


## 2. Проверка стилевых признаков и генерация

In [ ]:
STYLES = ["готический", "футуристический", "ретро"]
style_chain = ChatPromptTemplate.from_messages([
    ("system", 'Разбери описание сцены. Верни JSON {{"found_styles":[],"neutral_objects":[]}}. found_styles только из заданного списка; neutral_objects — объекты без этих стилевых признаков.'),
    ("human", "Стили: {styles}. Описание: {description}")
]) | llm | StrOutputParser()
random.seed(42)
style_scenes = [description, "Готический собор, футуристический автомобиль, ретро радиоприёмник. Рядом обычная скамейка."]
for index, scene in enumerate(style_scenes):
    result = json_result(style_chain.invoke({"styles": STYLES, "description": scene}))
    found = result["found_styles"]
    if all(style in found for style in STYLES):
        selected_style = random.choice(STYLES)
        prompt = f"{scene} Вся сцена в стиле {selected_style}."
    else:
        selected_style = "нейтральный"
        prompt = "Только эти предметы в нейтральном стиле: " + ", ".join(result["neutral_objects"])
    print("Анализ:", result, "Выбранный стиль:", selected_style)
    path = generate_image(prompt, f"lab07_styles_{index}.png")
    display(Image(filename=str(path), width=450))


## Вывод

Реальная Vision-модель описывает входное изображение. Цепочки определяют наличие объектов и стилей, а выбранная ветка формирует промпт генерации. Проверены обе ветки каждого условия.